# SageMaker Pipelines with MLflow

https://github.com/aws/amazon-sagemaker-examples/blob/main/sagemaker-mlflow/sagemaker_pipelines_mlflow.ipynb

## Run in root directory rather than shared/ for reduced s3 spend and better performance

## Setup environment

Import necessary libraries

In [1]:
import os

import sagemaker
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.function_step import step
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Declare some variables used later

In [2]:
sagemaker_session = sagemaker.session.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name

pipeline_name = "lmr-xgb-training-pipeline"
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")

# Mlflow (replace these values with your own)
tracking_server_arn = "arn:aws:sagemaker:us-east-1:575108933641:mlflow-tracking-server/lmr-tracking-server-5t7l23o0xvt99j-chws71x3trpelj-dev"
experiment_name = "lmr-sm-pipelines-experiment"

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Write `requirements` and `config` files that'll be used by the steps in our SageMaker Pipeline

In [3]:
%%writefile config.yaml
SchemaVersion: '1.0'
SageMaker:
  PythonSDK:
    Modules:
      RemoteFunction:
        # role arn is not required if in SageMaker Notebook instance or SageMaker Studio
        # Uncomment the following line and replace with the right execution role if in a local IDE
        # RoleArn: <replace the role arn here>
        InstanceType: ml.m5.xlarge
        Dependencies: ./requirements.txt
        IncludeLocalWorkDir: true
        CustomFileFilter:
          IgnoreNamePatterns: # files or directories to ignore
          - "*.ipynb" # all notebook files

Overwriting config.yaml


In [4]:
# Set path to config file
os.environ["SAGEMAKER_USER_CONFIG_OVERRIDE"] = os.getcwd()

In [5]:
%%writefile requirements.txt
scikit-learn
xgboost==1.7.6
s3fs==2024.12.0
sagemaker==2.254.1
pandas>=2.0.0
gevent
geventhttpclient
shap
matplotlib
fsspec
mlflow==2.13.2
sagemaker-mlflow==0.1.0
cloudpickle==3.1.2

Overwriting requirements.txt


## Define the SageMaker Pipeline

### Preprocessing Step

In [6]:
# Location of our dataset
file_name = "placeholder.csv"
input_path = f"s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/data/training/processed/{file_name}"

The breast cancer Wisconsin dataset contains column `id` which we do not use for training. The second column `diagnosis` is class label, and the label is represented using 'M' for Malignant class, and 'B' for Benign class. 

In the preprocessing step, we drop the column `id`, then split the dataset into three distinct sets: train, validation, and test set.

Note that `keep_alive_period_in_seconds` parameter in @step decorator indicates how many seconds we want to keep the instance alive, waiting to be reused for the next pipeline step execution. Setting this parameter speeds up the pipeline execution because we reduce the launching of new instances to execute pipeline steps.

In [7]:
# TODO: update preprocessing to match Skylar's notebook

random_state = 2023
label_column = "tlu_loss"

feature_names = [
    "soil","ppt","pdsi","vpd",
    "ndvi","lai","lst"
]


@step(
    name="DataPreprocessing",
    instance_type=instance_type,
)
def preprocess(
    raw_data_s3_path: str,
    output_prefix: str,
    experiment_name: str,
    run_name: str,
    test_size: float = 0.2,
) -> tuple:
    import mlflow
    import pandas as pd
    from sklearn.model_selection import train_test_split

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        with mlflow.start_run(run_name="DataPreprocessing", nested=True):
            df = pd.read_csv(raw_data_s3_path, header=None, names=feature_names)
            df.drop(columns="id", inplace=True)
            mlflow.log_input(
                mlflow.data.from_pandas(df, raw_data_s3_path, targets=label_column),
                context="DataPreprocessing",
            )

            train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[label_column])
            validation_df, test_df = train_test_split(
                test_df, test_size=0.5, stratify=test_df[label_column]
            )
            train_df.reset_index(inplace=True, drop=True)
            validation_df.reset_index(inplace=True, drop=True)
            test_df.reset_index(inplace=True, drop=True)

            train_s3_path = f"s3://{bucket}/{output_prefix}/train.csv"
            val_s3_path = f"s3://{bucket}/{output_prefix}/val.csv"
            test_s3_path = f"s3://{bucket}/{output_prefix}/test.csv"

            train_df.to_csv(train_s3_path, index=False)
            validation_df.to_csv(val_s3_path, index=False)
            test_df.to_csv(test_s3_path, index=False)

    return train_s3_path, val_s3_path, test_s3_path, experiment_name, run_id

### Training Step

We train a XGBoost model in this training step, using @step-decorated function with the S3 path of training and validation set, along with XGBoost hyperparameters. The S3 paths for both training and validation set is coming from the output of the previous step.

This could be replaced or supplemented with an option for selecting a different model architecture

In [8]:
use_gpu = False
param = dict(
    objective="reg:squarederror",
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.7,
    tree_method="gpu_hist" if use_gpu else "hist",  # Use GPU accelerated algorithm
)
num_round = 50


@step(
    name="ModelTraining",
    instance_type=instance_type,
)
def train(
    train_s3_path: str,
    validation_s3_path: str,
    experiment_name: str,
    run_id: str,
    param: dict = param,
    num_round: int = num_round,
):
    import mlflow
    import pandas as pd
    from xgboost import XGBClassifier

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelTraining", nested=True) as training_run:
            training_run_id = training_run.info.run_id
            mlflow.xgboost.autolog(
                log_input_examples=True,
                log_model_signatures=True,
                log_models=True,
                log_datasets=True,
                model_format="xgb",
            )

            # read data files from S3
            train_df = pd.read_csv(train_s3_path)
            validation_df = pd.read_csv(validation_s3_path)

            # create dataframe and label series
            y_train = (train_df.pop(label_column) == "M").astype("int")
            y_validation = (validation_df.pop(label_column) == "M").astype("int")

            xgb = XGBClassifier(n_estimators=num_round, **param)
            xgb.fit(
                train_df,
                y_train,
                eval_set=[(validation_df, y_validation)],
                early_stopping_rounds=5,
            )

        # return xgb
        return experiment_name, run_id, training_run_id

### Evaluation Step

In this step, we create a @step-decorated function to evaluate the trained XGBoost model on the test dataset.

In [9]:
@step(
    name="ModelEvaluation",
    instance_type=instance_type, 
)
def evaluate(
    test_s3_path: str,
    experiment_name: str,
    run_id: str,
    training_run_id: str,
) -> dict:
    import mlflow
    import pandas as pd

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelEvaluation", nested=True):
            test_df = pd.read_csv(test_s3_path)
            # test_df[label_column] = (test_df[label_column] == "M").astype("int")
            # Removing target processing step from example binary classification task
            model = mlflow.pyfunc.load_model(f"runs:/{training_run_id}/model")

            results = mlflow.evaluate(
                model=model,
                data=test_df,
                targets=label_column,
                model_type="regressor", # "classifier"
                evaluators=["default"],
            )
            return {"f1_score": results.metrics["f1_score"]}

### Model Registration

In this step, we create a @step-decorated function to register our XGBoost model. 

In [10]:
@step(
    name="ModelRegistration",
    instance_type=instance_type,
)
def register(
    pipeline_name: str,
    experiment_name: str,
    run_id: str,
    training_run_id: str,
):
    import mlflow

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelRegistration", nested=True):
            mlflow.register_model(f"runs:/{training_run_id}/model", pipeline_name)

## Creating the SageMaker Pipeline

We connect all defined pipeline `@step` functions into a multi-step pipeline. Then, we submit and execute the pipeline.

In [11]:
preprocessing_step = preprocess(
    raw_data_s3_path=input_path,
    output_prefix=f"{pipeline_name}/dataset",
    experiment_name=experiment_name,
    run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
)

training_step = train(
    train_s3_path=preprocessing_step[0],
    validation_s3_path=preprocessing_step[1],
    experiment_name=preprocessing_step[3],
    run_id=preprocessing_step[4],
)

# TODO: Update condition to be appropriate for regression task
conditional_register_step = ConditionStep(
    name="ConditionalRegister",
    conditions=[
        ConditionGreaterThanOrEqualTo(
            left=evaluate(
                test_s3_path=preprocessing_step[2],
                experiment_name=preprocessing_step[3],
                run_id=preprocessing_step[4],
                training_run_id=training_step[2],
            )["f1_score"],
            right=0.8,
        )
    ],
    if_steps=[
        register(
            pipeline_name=pipeline_name,
            experiment_name=preprocessing_step[3],
            run_id=preprocessing_step[4],
            training_run_id=training_step[2],
        )
    ],
    else_steps=[FailStep(name="Fail", error_message="Model performance is not good enough")],
)

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        instance_type,
    ],
    steps=[preprocessing_step, training_step, conditional_register_step],
)

sagemaker.config INFO - Fetched defaults config from location: /home/sagemaker-user
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [12]:
pipeline.upsert(role_arn=role)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-03-03 03:46:23,876 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-03-03-03-46-21-964/function
2026-03-03 03:46:24,114 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-03-03-03-46-21-964/arguments
2026-03-03 03:46:24,798 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpddybrq_5/requirements.txt'
2026-03-03 03:46:24,880 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-03-03-03-46-21-964/pre_exec_script_and

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-03-03 03:46:30,404 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-03-03-03-46-21-964/function
2026-03-03 03:46:30,777 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-03-03-03-46-21-964/arguments
2026-03-03 03:46:30,978 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpehgeprg2/requirements.txt'
2026-03-03 03:46:31,072 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-03-03-03-46-21-964/pre_exec_script_and_dependencie

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-03-03 03:46:32,840 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-03-03-03-46-21-964/function
2026-03-03 03:46:33,072 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-03-03-03-46-21-964/arguments
2026-03-03 03:46:33,292 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpebtjep_k/requirements.txt'
2026-03-03 03:46:33,396 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-03-03-03-46-21-964/pre_exec_script_and

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-03-03 03:46:35,164 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-03-03-03-46-21-964/function
2026-03-03 03:46:35,340 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-03-03-03-46-21-964/arguments
2026-03-03 03:46:35,519 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmp2gct5gza/requirements.txt'
2026-03-03 03:46:35,615 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-03-03-03-46-21-964/pre_exec_script_and_depen

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline',
 'ResponseMetadata': {'RequestId': '2532bb8e-e318-45b9-a459-17caca1f2562',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '2532bb8e-e318-45b9-a459-17caca1f2562',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '93',
   'date': 'Tue, 03 Mar 2026 03:46:35 GMT'},
  'RetryAttempts': 0}}

## Execute the SageMaker Pipeline

In [13]:
execution = pipeline.start()

In [14]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline/execution/0001gtnxao8s',
 'PipelineExecutionDisplayName': 'execution-1772509596305',
 'PipelineExecutionStatus': 'Executing',
 'CreationTime': datetime.datetime(2026, 3, 3, 3, 46, 36, 239000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 3, 3, 3, 46, 36, 239000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:575108933641:user-profile/d-lf2eivtg2b7i/c4083418-d061-7068-829e-604008d36196',
  'UserProfileName': 'c4083418-d061-7068-829e-604008d36196',
  'DomainId': 'd-lf2eivtg2b7i',
  'IamIdentity': {'Arn': 'arn:aws:sts::575108933641:assumed-role/datazone_usr_role_5t7l23o0xvt99j_5w5uuggwdfhu3r/SageMaker',
   'PrincipalId': 'AROAYLZZJ5AEZSW7N2N4C:SageMaker',
   'SourceIdentity': 'c4083418-d061-7068-829e-604008d36196'}},
 'LastModifiedBy': {'UserPr

In [15]:
execution.wait()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 execution.wait()                                                                             │
│   2                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/workflow/pipeline.py:989 in wait               │
│                                                                                                  │
│    986 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│    987 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│    988 │   │   )                                                                                 │
│ ❱  989 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│    990 │                                                                                         │
│    991 │   def result(self, step_name: str):                                                     │
│    992 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/waiter.py:58 in wait                            │
│                                                                                                  │
│    55 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    56 │   # method.                                                                              │
│    57 │   def wait(self, **kwargs):                                                              │
│ ❱  58 │   │   Waiter.wait(self, **kwargs)                                                        │
│    59 │                                                                                          │
│    60 │   wait.__doc__ = WaiterDocstring(                                                        │
│    61 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/waiter.py:378 in wait                           │
│                                                                                                  │
│   375 │   │   │   │   return                                                                     │
│   376 │   │   │   if current_state == 'failure':           

In [ ]:
execution.list_steps()